In [1]:
df_fuel = (
    spark.read
    .option("header", "true")
    .csv(
        "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/fuel_purchases.csv"
    )
)

display(df_fuel)
df_fuel.printSchema()

print(f"Source records: {df_fuel.count()}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7588384c-da52-45a2-9686-fdf79e64c8cd)

root
 |-- fuel_purchase_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- purchase_date: string (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)
 |-- gallons: string (nullable = true)
 |-- price_per_gallon: string (nullable = true)
 |-- total_cost: string (nullable = true)
 |-- fuel_card_number: string (nullable = true)

Source records: 196442


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

# Fuel Purchases Schema

fuel_schema = StructType([
    StructField("fuel_purchase_id", StringType(), True),
    StructField("trip_id", StringType(), True),
    StructField("truck_id", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("purchase_date", DateType(), True),
    StructField("location_city", StringType(), True),
    StructField("location_state", StringType(), True),
    StructField("gallons", DoubleType(), True),
    StructField("price_per_gallon", DoubleType(), True),
    StructField("total_cost", DoubleType(), True),
    StructField("fuel_card_number", StringType(), True)
])

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 4, Finished, Available, Finished, False)

In [3]:
landing_path = (
    "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
    "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/fuel_purchases.csv"
)

df_fuel = (
    spark.read
    .option("header", "true")
    .schema(fuel_schema)
    .csv(landing_path)
)

display(df_fuel)

df_fuel.printSchema()

print(f"Source records: {df_fuel.count()}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3fd4a12d-3783-4337-9bc9-9a3fd7ed6784)

root
 |-- fuel_purchase_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)
 |-- gallons: double (nullable = true)
 |-- price_per_gallon: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- fuel_card_number: string (nullable = true)

Source records: 196442


In [4]:
# 1. NULL primary key
null_fuel_purchase_ids = (
    df_fuel
    .filter(F.col("fuel_purchase_id").isNull())
    .count()
)

print(f"NULL fuel purchase IDs: {null_fuel_purchase_ids}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 6, Finished, Available, Finished, False)

NULL fuel purchase IDs: 0


In [5]:
# 2. Duplicate primary key
duplicate_fuel_purchase_ids = (
    df_fuel
    .groupBy("fuel_purchase_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicate fuel purchase IDs: {duplicate_fuel_purchase_ids}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 7, Finished, Available, Finished, False)

Duplicate fuel purchase IDs: 0


In [6]:
# 3. NULL purchase dates
null_purchase_dates = (
    df_fuel
    .filter(F.col("purchase_date").isNull())
    .count()
)

print(f"NULL purchase dates: {null_purchase_dates}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 8, Finished, Available, Finished, False)

NULL purchase dates: 0


In [7]:
# 4. Invalid gallons
invalid_gallons = (
    df_fuel
    .filter(
        F.col("gallons").isNull() |
        (F.col("gallons") <= 0)
    )
    .count()
)

print(f"Invalid gallons: {invalid_gallons}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 9, Finished, Available, Finished, False)

Invalid gallons: 0


In [8]:
# 5. Invalid fuel price
invalid_price = (
    df_fuel
    .filter(
        F.col("price_per_gallon").isNull() |
        (F.col("price_per_gallon") <= 0)
    )
    .count()
)

print(f"Invalid fuel price: {invalid_price}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 10, Finished, Available, Finished, False)

Invalid fuel price: 0


In [9]:
# 6. Invalid total cost
invalid_total_cost = (
    df_fuel
    .filter(
        F.col("total_cost").isNull() |
        (F.col("total_cost") < 0)
    )
    .count()
)

print(f"Invalid total cost: {invalid_total_cost}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 11, Finished, Available, Finished, False)

Invalid total cost: 0


In [10]:
# Refrential Integrity 
# Fuel Purchase → Trip
invalid_fuel_trip_ids = (
    df_fuel
    .filter(F.col("trip_id").isNotNull())
    .join(
        spark.table("bronze_trips").select("trip_id"),
        on="trip_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL trip IDs: {invalid_fuel_trip_ids}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 12, Finished, Available, Finished, False)

Invalid non-NULL trip IDs: 0


In [11]:
# Fuel Purchase → Truck
invalid_fuel_truck_ids = (
    df_fuel
    .filter(F.col("truck_id").isNotNull())
    .join(
        spark.table("bronze_trucks").select("truck_id"),
        on="truck_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL truck IDs: {invalid_fuel_truck_ids}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 13, Finished, Available, Finished, False)

Invalid non-NULL truck IDs: 0


In [12]:
# Fuel Purchase → Driver
invalid_fuel_driver_ids = (
    df_fuel
    .filter(F.col("driver_id").isNotNull())
    .join(
        spark.table("bronze_drivers").select("driver_id"),
        on="driver_id",
        how="left_anti"
    )
    .count()
)

print(f"Invalid non-NULL driver IDs: {invalid_fuel_driver_ids}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 14, Finished, Available, Finished, False)

Invalid non-NULL driver IDs: 0


In [13]:
df_fuel_bronze = (
    df_fuel
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("fuel_purchases.csv"))
)

display(df_fuel_bronze.limit(10))

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4085984c-5205-4d2e-a46f-eb57d6bde046)

In [14]:
df_fuel_bronze.createOrReplaceTempView("fuel_purchases_source")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 16, Finished, Available, Finished, False)

In [15]:
# Bronze Table
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_fuel_purchases (
    fuel_purchase_id STRING,
    trip_id STRING,
    truck_id STRING,
    driver_id STRING,
    purchase_date DATE,
    location_city STRING,
    location_state STRING,
    gallons DOUBLE,
    price_per_gallon DOUBLE,
    total_cost DOUBLE,
    fuel_card_number STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")

print("bronze_fuel_purchases table is ready.")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 17, Finished, Available, Finished, False)

bronze_fuel_purchases table is ready.


In [16]:
spark.sql("""
MERGE INTO bronze_fuel_purchases AS target

USING fuel_purchases_source AS source

ON target.fuel_purchase_id = source.fuel_purchase_id

WHEN MATCHED THEN
    UPDATE SET
        target.trip_id = source.trip_id,
        target.truck_id = source.truck_id,
        target.driver_id = source.driver_id,
        target.purchase_date = source.purchase_date,
        target.location_city = source.location_city,
        target.location_state = source.location_state,
        target.gallons = source.gallons,
        target.price_per_gallon = source.price_per_gallon,
        target.total_cost = source.total_cost,
        target.fuel_card_number = source.fuel_card_number,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        fuel_purchase_id,
        trip_id,
        truck_id,
        driver_id,
        purchase_date,
        location_city,
        location_state,
        gallons,
        price_per_gallon,
        total_cost,
        fuel_card_number,
        ingestion_timestamp,
        source_file
    )

    VALUES (
        source.fuel_purchase_id,
        source.trip_id,
        source.truck_id,
        source.driver_id,
        source.purchase_date,
        source.location_city,
        source.location_state,
        source.gallons,
        source.price_per_gallon,
        source.total_cost,
        source.fuel_card_number,
        source.ingestion_timestamp,
        source.source_file
    )
""")

print("Fuel Purchases Bronze MERGE completed successfully.")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 18, Finished, Available, Finished, False)

Fuel Purchases Bronze MERGE completed successfully.


In [17]:
bronze_fuel_count = (
    spark.table("bronze_fuel_purchases")
    .count()
)

print(f"Bronze fuel purchase records: {bronze_fuel_count}")

StatementMeta(, 0576c40a-1255-42a0-975f-454ccd3455a1, 19, Finished, Available, Finished, False)

Bronze fuel purchase records: 196442
